In [1]:
from nltk import word_tokenize
from nltk.corpus import stopwords
stops = stopwords.words('russian')
from nltk import sent_tokenize
import string
from collections import Counter
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

In [2]:
df = pd.read_csv(r"C:\Users\yuraz\Downloads\russian_novels_metadata_char.csv")
df

,author,title,subtitle,year,author_gender,pseudonym,genre,general_genre,genre1,genre2,...,char_эмигранты,char_придворная_знать,char_семейная_пара,char_рыцари,char_графы,char_слуги,char_врачи,char_ученые,char_types_count,has_char_info
0,"Эмин, Федор Александрович","Награждённая постоянность, или Приключения Лиз...",Сочинения Федора Эмина,1763,мужской,0,любовный,любовный,NaN,NaN,...,0,0,0,0,0,0,0,0,1,True
1,"Эмин, Федор Александрович","Непостоянная фортуна, или Похождения Мирамонда",NaN,1763,мужской,0,авантюрный,приключенческий,NaN,NaN,...,0,0,0,0,0,0,0,0,1,True
2,"Эмин, Федор Александрович","Приключения Фемистокла и разные политические, ...",NaN,1763,мужской,0,исторический,исторический,NaN,NaN,...,0,0,0,0,0,0,0,0,1,True
3,"Чулков, Михаил Дмитриевич","Пересмешник, или Славенские сказки",NaN,1766,мужской,0,NaN,NaN,авантюрный,сатирический,...,0,0,0,0,0,0,0,0,1,True
4,"Эмин, Федор Александрович",Письма Эрнеста и Доравры,NaN,1766,мужской,0,"философский, любовный",философский,NaN,NaN,...,0,0,0,0,0,0,0,0,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023,"Соловьев, Всеволод Сергеевич",Русские крестоносцы,повесть,1917,мужской,0,"дворяне, военные",исторический,NaN,NaN,...,0,0,0,0,0,0,0,0,1,True
2024,"Сухотин, Павел Сергеевич",Перчатка,"Записки русского кота, найденные Павлом Сухотиным",1917,мужской,0,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,1,True
2025,"Тимковский, Николай Иванович",В дворянской берлоге,Роман,1917,мужской,0,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,1,True
2026,"Тыркова, Ариандна Владимировна",Добыча,Роман,1917,женский,0,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,1,True


In [3]:
authors = df['author'].unique()
n = len(authors)//5

part1 = authors[0*n : 1*n]
part2 = authors[1*n : 2*n]
part3 = authors[2*n : 3*n]
part4 = authors[3*n : 4*n]
part5 = authors[4*n : ]

In [4]:
import time
url = "https://ru.wikipedia.org/w/api.php"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 YaBrowser/26.4.0.0 Safari/537.36',
}

authors_titles = {}
shit = []

def bio_parse(thing):
    found = {}
    missed = []
    for t in thing:
        if ', ' in t:
            query_name = t.split(', ')[1] + ' ' + t.split(', ')[0]
        else:
            query_name = t
        params = {
            'action': 'query',
            'list': 'search',
            'srsearch': query_name,
            'srlimit': 5,
            'format': 'json'
        }
        try:
            a = requests.get(url, headers=headers, params=params)
            b = a.json()
            if "query" in b and b['query']['search']:
                c = b['query']['search']
                found[query_name] = [x['title'] for x in c]
            else:
                missed.append(query_name)
        except Exception as e:
            missed.append(query_name)
            print(f'Ошибка на"{query_name}": {e}')
        time.sleep(2)
    return found, missed

In [5]:
titles1, missed1 = bio_parse(part1)

missed1, titles1

Ошибка на"Матвей Комаров": Expecting value: line 1 column 1 (char 0)
Ошибка на"Василий Васильевич Лазаревич": Expecting value: line 1 column 1 (char 0)
Ошибка на"Василий Алексеевич Левшин": Expecting value: line 1 column 1 (char 0)
Ошибка на"Наталья Алексеевна Неелова": Expecting value: line 1 column 1 (char 0)
Ошибка на"Михаил Михайлович Щербатов": Expecting value: line 1 column 1 (char 0)
Ошибка на"Иван Алексеевич Новиков": Expecting value: line 1 column 1 (char 0)
Ошибка на"Федор Лазинский": Expecting value: line 1 column 1 (char 0)
Ошибка на"Василий Михайлович Протопопов": Expecting value: line 1 column 1 (char 0)
Ошибка на"Николай Федорович Эмин": Expecting value: line 1 column 1 (char 0)
Ошибка на"Андрей Филиппов": Expecting value: line 1 column 1 (char 0)
Ошибка на"Дмитрий Николаевич Зиновьев": Expecting value: line 1 column 1 (char 0)
Ошибка на"Дмитрий Петрович Горчаков": Expecting value: line 1 column 1 (char 0)
Ошибка на"Василий Семенович Березайский": Expecting value: line 1

(['Иермонах Аполлос',
  'Матвей Комаров',
  'Василий Васильевич Лазаревич',
  'Василий Алексеевич Левшин',
  'Наталья Алексеевна Неелова',
  'Михаил Михайлович Щербатов',
  'Иван Алексеевич Новиков',
  'Федор Лазинский',
  'Василий Михайлович Протопопов',
  'Николай Федорович Эмин',
  'Андрей Филиппов',
  'Дмитрий Николаевич Зиновьев',
  'Василий Александрович Иноградский',
  'Дмитрий Петрович Горчаков',
  'Василий Семенович Березайский',
  'Яков Андреевич Галинковский',
  'Иван Ипатович Запольский',
  'Александр Ефимович Измайлов',
  'Николай Иванович Гнедич',
  'Максим Петрович Башурлов',
  'Николай Назарьевич Муравьев',
  'Михаил Степанович (Стефанович) Бранкевич',
  'Мария Евграфовна Извекова',
  'Федор Петрович Львов',
  'Василий Трофимович Нарежный',
  'Иван Сергеевич Георгиевский',
  'Александр Фомич Вельтман',
  'Иван Никитич Глухарев',
  'Николай Иванович Греч',
  'Иван Гавриилович Гурьянов',
  'Иван Тимофеевич Калашников',
  'Иван Иванович Лажечников',
  'Варвара Семеновна Ми

In [ ]:
for test in range(5):
    if len(missed1) == 0:
        break
    still_missed = []
    for t in list(missed1):
        if ', ' in t:
            query_name =t.split(', ')[0]
        else:
            query_name = t
        params = {
            'action': 'query',
            'list': 'search',
            'srsearch': query_name,
            'srlimit': 5,
            'format': 'json'
        }
        try:
            a = requests.get(url, headers=headers, params=params)
            b = a.json()
            if "query" in b and b['query']['search']:
                c = b['query']['search']
                titles1[query_name] = [x['title'] for x in c]
            else:
                still_missed.append(query_name)
        except Exception as e:
            still_missed.append(query_name)
            print(f'Ошибка на"{query_name}": {e}')
        time.sleep(2)
    missed1 = still_missed
missed1
titles1

In [6]:
titles2, missed2 = bio_parse(part2)

Ошибка на"Федор Богданович Миллер": Expecting value: line 1 column 1 (char 0)
Ошибка на"Василий Иванович Мирошевский": Expecting value: line 1 column 1 (char 0)
Ошибка на"Петр Алексеевич Салманов": Expecting value: line 1 column 1 (char 0)
Ошибка на"Александр Павлович Башуцкий": Expecting value: line 1 column 1 (char 0)
Ошибка на"Федор Федорович Корф": Expecting value: line 1 column 1 (char 0)
Ошибка на"Михаил Юрьевич Лермонтов": Expecting value: line 1 column 1 (char 0)
Ошибка на"Нил Алексеевич Мышицкий": Expecting value: line 1 column 1 (char 0)
Ошибка на"Владимир Федорович Одоевский": Expecting value: line 1 column 1 (char 0)
Ошибка на"Софья Алексеевна Закревская": Expecting value: line 1 column 1 (char 0)
Ошибка на"Нестор Васильевич Кукольник": Expecting value: line 1 column 1 (char 0)
Ошибка на"Осип Иванович Сенковский": Expecting value: line 1 column 1 (char 0)
Ошибка на"Федор Фан-Дим": Expecting value: line 1 column 1 (char 0)
Ошибка на"Борис Михайлович Федоров": Expecting value

In [ ]:
for test in range(5):
    if len(missed2) == 0:
        break
    still_missed2 = []
    for t in list(missed2):
        if ', ' in t:
            query_name =t.split(', ')[0]
        else:
            query_name = t
        params = {
            'action': 'query',
            'list': 'search',
            'srsearch': query_name,
            'srlimit': 5,
            'format': 'json'
        }
        try:
            a = requests.get(url, headers=headers, params=params)
            b = a.json()
            if "query" in b and b['query']['search']:
                c = b['query']['search']
                titles2[query_name] = [x['title'] for x in c]
            else:
                still_missed2.append(query_name)
        except Exception as e:
            still_missed2.append(query_name)
            print(f'Ошибка на"{query_name}": {e}')
        time.sleep(2)
    missed2 = still_missed2
still_missed2
titles2

In [ ]:
titles3, missed3 = bio_parse(part3)

Ошибка на"Дмитрий Платонович Ломачевский": Expecting value: line 1 column 1 (char 0)
Ошибка на"Болеслав Михайлович Маркевич": Expecting value: line 1 column 1 (char 0)
Ошибка на"Яков Петрович Полонский": Expecting value: line 1 column 1 (char 0)
Ошибка на"Александр Алексеевич Соколов": Expecting value: line 1 column 1 (char 0)
Ошибка на"Василий Иванович Кельсиев": Expecting value: line 1 column 1 (char 0)
Ошибка на"Иннокентий Васильевич Омулевский": Expecting value: line 1 column 1 (char 0)
Ошибка на"Михаил Евграфович Салтыков-Щедрин": Expecting value: line 1 column 1 (char 0)
Ошибка на"Сергей Николаевич Худеков": Expecting value: line 1 column 1 (char 0)
Ошибка на"Николай Александрович Чаев": Expecting value: line 1 column 1 (char 0)
Ошибка на"Александра Никитична Анненская": Expecting value: line 1 column 1 (char 0)
Ошибка на"П. Ближнев": Expecting value: line 1 column 1 (char 0)
Ошибка на"Леонид Петрович Блюммер": Expecting value: line 1 column 1 (char 0)
Ошибка на"Григорий Исаакови

In [ ]:
for test in range(5):
    if len(missed3) == 0:
        break
    still_missed3 = []
    for t in list(missed3):
        if ', ' in t:
            query_name =t.split(', ')[0]
        else:
            query_name = t
        params = {
            'action': 'query',
            'list': 'search',
            'srsearch': query_name,
            'srlimit': 5,
            'format': 'json'
        }
        try:
            a = requests.get(url, headers=headers, params=params)
            b = a.json()
            if "query" in b and b['query']['search']:
                c = b['query']['search']
                titles3[query_name] = [x['title'] for x in c]
            else:
                still_missed3.append(query_name)
        except Exception as e:
            still_missed3.append(query_name)
            print(f'Ошибка на"{query_name}": {e}')
        time.sleep(2)
    missed3 = still_missed3
still_missed3
titles3

In [ ]:
titles4, missed4 = bio_parse(part4)

In [ ]:
for test in range(5):
    if len(missed4) == 0:
        break
    still_missed4 = []
    for t in list(missed4):
        if ', ' in t:
            query_name =t.split(', ')[0]
        else:
            query_name = t
        params = {
            'action': 'query',
            'list': 'search',
            'srsearch': query_name,
            'srlimit': 5,
            'format': 'json'
        }
        try:
            a = requests.get(url, headers=headers, params=params)
            b = a.json()
            if "query" in b and b['query']['search']:
                c = b['query']['search']
                titles4[query_name] = [x['title'] for x in c]
            else:
                still_missed4.append(query_name)
        except Exception as e:
            still_missed4.append(query_name)
            print(f'Ошибка на"{query_name}": {e}')
        time.sleep(2)
    missed4 = still_missed4
still_missed4
titles4

In [ ]:
titles5, missed5 = bio_parse(part5)

In [ ]:
for test in range(5):
    if len(missed5) == 0:
        break
    still_missed5 = []
    for t in list(missed5):
        if ', ' in t:
            query_name =t.split(', ')[0]
        else:
            query_name = t
        params = {
            'action': 'query',
            'list': 'search',
            'srsearch': query_name,
            'srlimit': 5,
            'format': 'json'
        }
        try:
            a = requests.get(url, headers=headers, params=params)
            b = a.json()
            if "query" in b and b['query']['search']:
                c = b['query']['search']
                titles5[query_name] = [x['title'] for x in c]
            else:
                still_missed5.append(query_name)
        except Exception as e:
            still_missed5.append(query_name)
            print(f'Ошибка на"{query_name}": {e}')
        time.sleep(2)
    missed5 = still_missed5
still_missed5
titles5

In [ ]:
t = 'Свиньин, Павел Петрович'
if ', ' in t:
    query_name = t.split(', ')[1] + ' ' + t.split(', ')[0]
else:
    query_name = t
query_name

In [ ]:
all_titles = {}
all_titles.update(titles1)
all_titles.update(titles2)
all_titles.update(titles3)
all_titles.update(titles4)
all_titles.update(titles5)
titles_list = []
for x in all_titles:
    y = all_titles[x]
    titles_list.append(y[0])

In [ ]:
def bio_text_parse(theme):
    authors_bio = {}
    shit = []
    N = 15
    for i in range(0, len(theme), N):
        batch = theme[i : i+N]
        titles_str = '|'.join(batch) 
        params = {
            'action': 'query',
            'prop': 'extracts',
            'titles': titles_str,
            'explaintext': True,
            'exlimit': 'max',
            'format': 'json'
        }
        try:
            a = requests.get(url, headers=headers, params=params)
            b = a.json()
            if "query" in b and b['query']['pages']:
                pages = b['query']['pages']
                for page in pages.values():
                    if 'extract' in page:
                        authors_bio[page['title']] = page['extract']
        except Exception as e:
            shit.append(titles_str)
            print(f'Ошибка на пачке:{titles_str[:80]} | {e}')
        time.sleep(1)
    return authors_bio, shit

In [ ]:
authors_bio, shit = bio_text_parse(titles_list)
authors_bio

In [ ]:
print(len(titles_list))
print(len(authors_bio))
print(len(shit))

In [ ]:
df.columns

In [ ]:
df.head(3)

In [ ]:
all_titles_fixed = {}
authors = df['author'].unique()
for author in authors:
    if ', ' in author:
        a = author.split(', ')[1] + ' ' + author.split(', ')[0]
    else:
        a = author
    if a in all_titles:
        all_titles_fixed[author] = all_titles[a]

In [ ]:
author = list(all_titles_fixed.keys())[0]
print(author)
print(df[df['author'] == author]['title'].tolist())

In [ ]:
th = 'Эмин, Федор Александрович'
a = all_titles_fixed[author]
print('Cand', a)
for x in a:
    params = {
    'action': 'query',
    'prop': 'extracts',
    'titles': x,
    'explaintext': True,
    'exlimit': 'max',
    'format': 'json'
    }
    c = requests.get(url, headers=headers, params=params)
    d = c.json()
    text = list(d['query']['pages'].values())[0].get('extract', '')
    text_lower = text.lower()
    est_marker = any(m in text_lower for m in mark)
    print(f'{x[:40]} | длина текста: {len(text)} | mark: {est_marker}')

In [ ]:
mark = ['писатель', 'литератор', 'поэт', 'библиография', 'сочинения', 'прозаик', 'драматург']
def derive(autho):
    biography = {}
    trash = []
    for author in autho:
        a = autho[author]
        b = []
        for x in a:
            params = {
                'action': 'query',
                'prop': 'extracts',
                'titles': x,
                'explaintext': True,
                'exlimit': 'max',
                'format': 'json'
            }
            text = ''
            for t in range(3):
                try:
                    c = requests.get(url, headers=headers, params=params)
                    d = c.json()
                    text = list(d['query']['pages'].values())[0].get('extract', '')
                    if text:
                        break
                except:
                    pass
                time.sleep(0,2)

            text_lower = text.lower()
            if any(m in text_lower for m in mark):
                b.append((x, text))
        if len(b) == 1:
            biography[author] = b[0][1]
        elif len(b) > 1:
            e = df[df['author'] == author]['title'].tolist()
            f = [str(p).split(',')[0].split(' или ')[0] for p in e]
            g = None
            for x, text in b:
                if any(k in text for k in f if k and k != 'nan'):
                    g = text
                    break
            if g:
                biography[author] = g
            else:
                biography[author] = b[0][1]
        else:
            trash.append(author)
    return biography, trash

In [ ]:
def freak_try(start, end):
    test = dict(list(all_titles_fixed.items())[start:end])
    bio_test, trash_test = derive(test)

    for author in bio_test:
        print(author, '->', bio_test[author][:80])
    print('BS:', trash_test)
    print('Good:', bio_test)

In [ ]:
freak_try(0, 100)

In [ ]:
freak_try(100, 200)

In [ ]:
freak_try(200, 300)

In [ ]:
freak_try(300, 400)

In [ ]:
freak_try(400, 500)

In [ ]:
freak_try(500, 600)